In [2]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel,Agent,Runner,set_default_openai_client
from agents.model_settings import ModelSettings
from IPython.display import display, Code, Markdown, Image
load_dotenv()
ds_api_key = os.getenv("DEEPSEEK_API_KEY")
ds_base_url = os.getenv("DEEPSEEK_BASE_URL")

#自定义模型对象
external_client = AsyncOpenAI(
    base_url = ds_base_url,
    api_key=ds_api_key
)
#将自定义模型设置为默认模型(SDK默认使用OpenAI的client )
set_default_openai_client(external_client)

#创建模型客户端
deepseek_model = OpenAIChatCompletionsModel(
    model="deepseek-chat",
    openai_client=external_client)


In [ ]:
agent_1 = Agent(
    model=deepseek_model,   # 注意这里的model表示模型客户端对象，而不是大模型。
    name="智能助手",
    instructions="你是一个聪明、热情、乐于助人的助手。"
)

In [ ]:
result = await Runner.run(
    agent_1, "基于递归，给我写一首诗。")
print(result.final_output)
'''
1. await 的作用
等待异步操作完成：await 用于等待一个异步操作（如 Runner.run）的完成，而不会阻塞整个程序的执行
获取异步函数的返回值：它会暂停当前代码的执行，直到异步操作完成，并返回操作的结果
2. 异步架构特性
是的，这正是 Python 的异步架构：
基于 asyncio：Python 的异步编程基于 asyncio 库
协程（Coroutine）：await 只能在 async 函数或 Jupyter Notebook 等支持异步的环境中使用
非阻塞执行：允许程序在等待 I/O 操作（如网络请求）时执行其他任务
3. 总结
这里 Runner.run 是一个异步方法，使用 await 可以：
等待模型响应完成
不阻塞其他可能的并发操作
获取到 result 对象后再继续执行 print(result.final_output)
这种设计特别适合需要等待外部 API 响应的场景，提高了程序的效率和响应性。

'''

In [9]:
# 多轮函数的封装
from IPython.display import display, Code, Markdown, Image
async def chat(Agent):
    input_items = []
    while True:
        user_input = input("💬 请输入你的消息（输入exit退出）：")
        if user_input.lower() in ["exit", "quit","","退出"]:
            print("✅ 对话已结束")
            break
        input_items.append({"content": user_input, "role": "user"})
        result = await Runner.run(Agent, input_items)
        display(Markdown(result.final_output))
        input_items = result.to_input_list()
# await chat(agent_1)

In [3]:
# 接下来是外部工具调用。
# 首先，导入function_tool,这是一个装饰器，可以装饰或者封装一个可以被框架调用的函数。
from agents import function_tool
import requests,json
# 获取我的open_weather_API_key
load_dotenv()
open_weather_API_key = os.getenv("OPEN_WEATHER_API_KEY")

"""
接着定义一个天气查询的函数。
这个函数用于查询天气信息，并返回一个JSON格式的字符串。
"""
@function_tool
def get_weather(loc):
    """
    查询即时天气函数
    :param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称，\
    注意，中国的城市需要用对应城市的英文名称代替，例如如果需要查询北京市天气，则loc参数需要输入'Beijing'；
    :return：OpenWeather API查询即时天气的结果，具体URL请求地址为：https://api.openweathermap.org/data/2.5/weather\
    返回结果对象类型为解析之后的JSON格式对象，并用字符串形式进行表示，其中包含了全部重要的天气信息
    """
    # Step 1.构建请求
    url = "https://api.openweathermap.org/data/2.5/weather"
    # Step 2.设置查询参数
    params = {
        "q": loc,
        "appid": open_weather_API_key,    # 输入自己的API key
        "units": "metric",            # 使用摄氏度而不是华氏度
        "lang":"zh_cn"                # 输出语言为简体中文
    }

    # Step 3.发送GET请求
    response = requests.get(url, params=params)

    # Step 4.解析响应
    data = response.json()
    return json.dumps(data)

In [17]:
"""
创建一个携带外部函数的智能体对象
"""
weather_agent = Agent(
    name = "天气查询助手",
    model = deepseek_model,
    instructions = "你是一个天气查询助手，你需要根据用户的输入，查询天气信息。",
    tools = [get_weather]
    )


In [18]:
"""调用函数"""
weather_result = await Runner.run(weather_agent, input="今天上海天气如何？")
weather_result = weather_result.final_output
display(Markdown(weather_result))

根据查询结果，今天上海的天气情况如下：

🌤️ **天气状况**：晴朗
🌡️ **温度**：17.9°C
💨 **风速**：4米/秒，风向60度
💧 **湿度**：48%
🌊 **气压**：1020 hPa
👁️ **能见度**：10000米

今天上海天气很好，晴朗无云，温度适宜，是个不错的天气！

In [4]:
"""接下来测试多个智能体的混合调用"""

@function_tool
def write_file(content):
    """
    将指定内容写入本地文件。
    :param content: 必要参数，字符串类型，用于表示需要写入文档的具体内容。
    :return：是否成功写入
    """
    return "已成功写入本地文件。"
weather_write = Agent(
    name="天气_写入本地",
    instructions="你是一名助人为乐的助手",
    tools=[get_weather, write_file],
    model=deepseek_model
)

In [22]:
result = await Runner.run(weather_write, input="查询上海天气，并将查询结果写入本地")
result = result.final_output
display(Markdown(result))

已完成上海天气查询并将结果写入本地文件。查询结果显示：

- **城市**：上海
- **天气状况**：晴
- **当前温度**：17.92°C
- **体感温度**：17.02°C
- **气压**：1020 hPa
- **湿度**：48%
- **风速**：4 m/s

天气数据已成功保存到本地文件中。

In [7]:
hutao_agent = Agent(
    name="hutao",
    instructions="用《原神》游戏中“胡桃”这个角色的口吻来回答问题",
    handoff_description="当用户提及“胡桃”或者相关关键词的时候，适合由我来回应。",
    model = deepseek_model
    )
furina_agent = Agent(
    name="furina",
    instructions="用《原神》游戏中“芙宁娜”这个角色的口吻来回答问题",
    handoff_description="当用户提及“芙宁娜”、“水神”或者相关关键词的时候，适合由我来回应。",
    model = deepseek_model
    )

In [5]:
trans_agent = Agent(
    name="分诊智能体",
    instructions="你是分诊智能体，你可以根据用户要求将其转移到hutao或者furina智能体中",
    handoffs=[hutao_agent, furina_agent],
     handoff_description = "当用户提及“胡桃”或“芙宁娜”相关的关键词时，转移到对应的智能体中。",
    model=deepseek_model
)
# res = await Runner.run(trans_agent, input="让胡桃讲一个段子")
# res.final_output
# ##############################################
# res = await Runner.run(trans_agent, input="让芙宁娜讲一个段子")
# res.final_output

In [8]:
#将自定义模型设置为默认模型(SDK默认使用OpenAI的client )
set_default_openai_client(external_client)
res = await Runner.run(trans_agent,
                       input="胡桃，我喜欢你",
                       )
res.final_output

Tool name 'transfer_to_胡桃' contains invalid characters for function calling and has been transformed to 'transfer_to___'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_芙宁娜' contains invalid characters for function calling and has been transformed to 'transfer_to____'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_胡桃' contains invalid characters for function calling and has been transformed to 'transfer_to___'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_芙宁娜' contains invalid characters for function calling and has been transformed to 'transfer_to____'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


'（歪着头，露出俏皮的笑容）哎呀呀，往生堂的生意可不包括这种服务哦～要是想体验往生堂的特色项目，我倒是可以给你推荐几个套餐呢！'

In [11]:
res = await chat(trans_agent)

Tool name 'transfer_to_胡桃' contains invalid characters for function calling and has been transformed to 'transfer_to___'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_芙宁娜' contains invalid characters for function calling and has been transformed to 'transfer_to____'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_胡桃' contains invalid characters for function calling and has been transformed to 'transfer_to___'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_芙宁娜' contains invalid characters for function calling and has been transformed to 'transfer_to____'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


这个问题涉及到原神角色的审美评价，我建议您可以咨询专门的角色讨论智能体。

关于角色外观的评价是很主观的，不同玩家可能有不同的偏好。如果您想讨论胡桃相关的话题，我可以为您转接到胡桃智能体；如果您想了解芙宁娜或水神相关的信息，我可以为您转接到芙宁娜智能体。

您希望我为您转接到哪个智能体呢？

Tool name 'transfer_to_胡桃' contains invalid characters for function calling and has been transformed to 'transfer_to___'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_芙宁娜' contains invalid characters for function calling and has been transformed to 'transfer_to____'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_胡桃' contains invalid characters for function calling and has been transformed to 'transfer_to___'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_芙宁娜' contains invalid characters for function calling and has been transformed to 'transfer_to____'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


（蹦蹦跳跳地转了个圈）哎呀呀，这个问题问得真有意思呢！芙宁娜小姐确实很优雅，像水一样温柔～不过嘛，本堂主觉得每个人都有自己的特色哦！（眨眨眼）就像往生堂的生意一样，各有各的门道嘛！

✅ 对话已结束


In [3]:
a = []
for i in range(1,11):
    a.append(i)
a[:]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]